<a id="member-m3"></a>
## M3 · 贝叶斯学习器与方法核查
**负责范围：** §4及紧随其后的检查与验收。

**输入 → 输出：** M2的特征/策略接口与`DEV_OUT` → `Selector`、`replay`、方法公式及反馈核查结论。

**依赖与交接：** 先执行M1/M2；本模块核查需要开发结果。完成后把可调用学习器和方法说明交M4。

### 4 · M3：解释并核查两个贝叶斯学习器
**目标：** 把课程推导对应到代码，说明不确定性如何推动探索；不要求发明新算法。

**TS：** 每臂正确率 $p_a\sim\mathrm{Beta}(1,1)$；观测该臂S次正确、F次错误后为 $\mathrm{Beta}(1+S,1+F)$。每轮采样正确率并减已知成本选臂；仅更新所选臂。手算检查：答对2次、答错1次应得Beta(3,2)，后验均值0.6。

**LaplaceCTS：** $x=(1,\mathbf1[项目名],\mathbf1[its])$，每臂 $P(Y=1|x,w_a)=\sigma(x^T w_a)$，先验 $w_a\sim N(0,I)$。MAP最小化logistic负对数似然加 $\|w_a\|^2/2$；协方差近似为 $(I+X^TWX)^{-1}$。从近似后验采样参数，再减成本选臂。

每臂累计5次新反馈才重拟合一次，未到5次用上一后验（初始为先验）；每条新对话重新初始化学习器。开发集不用于预训练正式对话的参数。

**课程：** Lecture 2 pp.16–22、49–50；Lecture 6 pp.11–15、27–30。这是课程概念组合后的项目实现，不声称课件直接给出了完整LaplaceCTS算法。

**边界：** 当前模板中“项目名”和“its”互补，与截距线性相关；高斯先验使Hessian可逆，但不宜解释单个系数的独立效应。特征被设计为可识别题型。TS也知道各臂当前成本，“不看题”准确地说是其成功率模型不使用问题特征。

RuleRouter对指代题用短窗，其他题用强检索，是人工规则参考，不是理论上限。DTS只保留开发γ表；UCB/Markov在附录，不纳入主结论。

Suppose $p_a$ is the success rate for arm $a$.
We choose beta(1,1) as our prior of $p_a$. Hence $p_a \sim \text{Beta}(1,1)$. 

After observing $s_a$ correct answers and $f_a$ incorrect answers, the posterior distribution is $p_a|D_a \sim \text{Beta}(1+s_a, 1+f_a)$.

### Method 1a
We use Thompson Sampling to sample $p_a$ and choose the arm that maximize the utility function
$$a^*=\arg\max_{a}(\tilde{p}_a-\frac{0.5C_{a}}{512})$$

### Method 1b
We can also use Discounted Thompson Sampling to sample $p_a$. It can give more weight to the recent performance and reduce the impact of the historical performance.

Assume $\gamma$ is the discounted factor. Hence we update the posterior distribution as follows:
$$p_a|D_a\sim\text{Beta}(1+s_a\gamma,1+f_a\gamma)$$
$$a^*=\arg\max_{a}(\tilde{p}_a-\frac{0.5C_{a}}{512})$$

### Method 2
Another possible method is that we choose the arm based on the context of the question enquired. 

First of all, we described the current question using a feature vector:

$$x=[1,I(\text{feature 1}),I(\text{feature 2}),...]$$

For example, $x = [1,I(\text{contains project name}),I(\text{contains "its"})]$

If the question is "What is its current dataset?", then $x=[1,0,1]$.

We model the estimated probability of getting a correct answer based on the arm $a$ chosen by a sigmoid function:

$$P(\text{correct answer}|x,a)=\sigma(x^\top w_a)$$

where $w_a$ is the weight vector of the arm $a$ that we need to be learned.

To sample $w_a$, we need to find out the distribution of $p(w_a|D_a)$. Note that

$$p(w_a|D_a)\propto p(w_a)\prod_{i\in D_a}\sigma(x^\top w_a)^{y_i}[1-\sigma(x^\top w_a)]^{1-y_i}$$

where $y_i=1$ when the $i$ th answer is correct and $y_i=0$ otherwise. Also we have a prior of $w_a\sim\mathcal N(0,I)$.

This distribution has no closed form. Hence we use laplace approximation to estimate the distribution of $w_a|D_a$:

First we find $\log p(w_a|D_a)$:

$$L(w_a)=\log p(w_a|D_a)=\log p(w_a) + \sum_{i\in D_a}y_i\log(\sigma(x^\top w_a))+(1-y_i)\log(1-\sigma(x^\top w_a))$$

$$=-\frac{1}{2}w_a^\top w_a + \sum_{i\in D_a}[y_i\log(\frac{e^{x^\top w_a}}{1+e^{x^\top w_a}})+(1-y_i)\log(\frac{1}{1+e^{x^\top w_a}})] + \text{constant}$$

$$=-\frac{1}{2}w_a^\top w_a + \sum_{i\in D_a}[y_i(x^\top w_a-\log(1+e^{x^\top w_a}))+(1-y_i)(-\log(1+e^{x^\top w_a}))] + \text{constant}$$

$$=-\frac{1}{2}w_a^\top w_a + \sum_{i\in D_a}[y_ix^\top w_a-\log(1+e^{x^\top w_a})] + \text{constant}$$

Then we find the MAP estimate $\hat w_a$ by finding where the gradient of $\log p(w_a|D_a)$ equal to $0$.

$$\nabla_{w_a}L(w)=-w_a+\sum_{i\in D_a}[y_ix-\frac{e^{x^\top w_a}}{1+e^{x^\top w_a}}x]$$

$$=-w_a+\sum_{i\in D_a}[y_i-\sigma({x^\top w_a})]x=0$$

Hence 

$$\hat w_a+\sum_{i\in D_a}[\sigma({x^\top \hat w_a})-y_i]x=0$$

The equation does not have a closed form solution, hence we use BFGS to estimate $\hat w_a$.

Also, the Hessian matrix $\mathbf{A}$

$$=-\nabla\nabla_{w_a}L(w_a)$$

$$=I+\sum_{i\in D_a}[\sigma(x^\top w_a)(1-\sigma(x^\top w_a))xx^\top]$$

Therefore, $p(\tilde{w}_a|D_a) \sim \mathcal N(\tilde{w}_{a,\text{MAP}},\mathbf{A}^{-1})$. We can sample $w_a$ from this multivariate normal distribution.

Hence, we can calculate $\tilde{p}_a=\sigma(x^\top \tilde{w_a})$. 

The $w_a$ is updated for each 5 observation for the arm $a$.

In [ ]:
class Selector:
    def __init__(self,name,seed,gamma=1.):
        assert name in ARMS+['TS','DTS','LaplaceCTS','RuleRouter','UCB'], name
        self.name=name; self.rng=np.random.default_rng(seed); self.gamma=gamma
        self.s=np.zeros(len(ARMS));self.f=self.s.copy();self.t=0
        self.X=[[] for _ in ARMS];self.Y=[[] for _ in ARMS]
        self.mu=np.zeros((len(ARMS),3));self.cov=np.tile(np.eye(3)/CFG['prior_precision'],(len(ARMS),1,1))
    def choose(self,q,cost):
        self.t+=1
        if self.name in ARMS:return ARMS.index(self.name)
        if self.name=='RuleRouter':return 0 if features(q)[2] and not features(q)[1] else 3
        if self.name=='LaplaceCTS':
            p=np.array([expit(features(q)@self.rng.multivariate_normal(self.mu[k],self.cov[k])) for k in range(len(ARMS))])
        elif self.name=='UCB':
            n=self.s+self.f
            if (n==0).any():return int(np.flatnonzero(n==0)[0])
            p=self.s/n+np.sqrt(2*np.log(self.t)/n)
        else:
            if self.name=='DTS': self.s*=self.gamma;self.f*=self.gamma
            p=self.rng.beta(1+self.s,1+self.f)
        return int(np.argmax(p-CFG['lam']*cost/CFG['cref']))
    def update(self,k,y,q):
        self.s[k]+=y;self.f[k]+=1-y
        if self.name!='LaplaceCTS':return
        self.X[k].append(features(q)); self.Y[k].append(y)
        if len(self.Y[k])%CFG['refit_every']:return
        X=np.array(self.X[k]);Y=np.array(self.Y[k])
        def fg(w):
            z=X@w; return np.logaddexp(0,z).sum()-Y@z+.5*CFG['prior_precision']*w@w, X.T@(expit(z)-Y)+CFG['prior_precision']*w
        fit=minimize(fg,self.mu[k],jac=True,method='BFGS',options={'gtol':1e-6,'maxiter':100})
        if not fit.success and np.linalg.norm(fit.jac)>1e-3:raise RuntimeError(fit.message)
        self.mu[k]=fit.x;p=expit(X@fit.x)
        self.cov[k]=np.linalg.inv(CFG['prior_precision']*np.eye(3)+X.T@((p*(1-p))[:,None]*X))

def replay(frame,name,alg_seed=17,gamma=1.):
    # 评估器拥有全臂反事实；choose仍只得到q/cost，update只得到选中y。
    selector=Selector(name,alg_seed,gamma); rows=[]
    expected=pd.MultiIndex.from_product([sorted(frame.t.unique()),ARMS],names=['t','arm'])
    g=frame.set_index(['t','arm'])
    assert g.index.is_unique and len(g)==len(expected)
    g=g.reindex(expected); assert not g.isna().any().any()
    y=g.y.to_numpy().reshape(-1,len(ARMS)); cost=g.cost.to_numpy().reshape(-1,len(ARMS))
    outdated=g.outdated.to_numpy().reshape(-1,len(ARMS))
    coverage=g.coverage.to_numpy().reshape(-1,len(ARMS))
    first=g.iloc[::len(ARMS)]
    for j,(t,q) in enumerate(zip(sorted(frame.t.unique()),first.question)):
        k=selector.choose(q,cost[j].copy())
        selected_y=int(y[j,k]); selector.update(k,selected_y,q)
        rows.append(dict(seed=int(first.seed.iloc[j]),scenario=first.scenario.iloc[j],t=int(t),method=name,
            alg_seed=alg_seed,arm=ARMS[k],y=selected_y,cost=int(cost[j,k]),
            utility=selected_y-CFG['lam']*cost[j,k]/CFG['cref'],outdated=int(outdated[j,k]),coverage=int(coverage[j,k])))
    return pd.DataFrame(rows)
# Core invariants: gamma=1 DTS exactly reproduces TS; only selected counts increase.
tiny=DEV_OUT[DEV_OUT.seed==DEV[0]['seed']]
a=replay(tiny,'TS');b=replay(tiny,'DTS',gamma=1.)
assert a.arm.tolist()==b.arm.tolist()
s=Selector('TS',0);s.update(2,1,'What is its dataset?'); assert np.array_equal(s.s,[0,0,1,0])
assert not any(x in public_view(DEV[0],DEV[0]['probes'][0]) for x in ['gold','state','qtype'])
print('TS/DTS identity, selected-only updates and public-view checks passed.')

TS/DTS identity, selected-only updates and public-view checks passed.


In [ ]:
# M3：检查的是反馈可见性与数值后验，而非“方法一定胜出”。
q_check='What is its current dataset?'
s_check=Selector('LaplaceCTS',17)
for y_check in [1,1,0,1,0]: s_check.update(1,y_check,q_check)
assert [len(y) for y in s_check.Y]==[0,5,0,0]
assert np.allclose(s_check.mu[[0,2,3]],0)
assert np.linalg.eigvalsh(s_check.cov[1]).min()>0
assert np.allclose(s_check.cov[1],s_check.cov[1].T)
assert np.allclose(features(q_check),[1,0,1])
print('LaplaceCTS selected-only feedback, positive-definite covariance, feature checks passed.')

LaplaceCTS selected-only feedback, positive-definite covariance, feature checks passed.


### M3 · 手算、推导与实现对应
Beta(1,1)连续观测1、1、0后为Beta(3,2)，均值3/5；成本惩罚不作为“失败次数”，只在选择时减去已知成本。

MAP目标为 $L(w)=\sum_i[\log(1+e^{x_i^Tw})-y_ix_i^Tw]+\frac12w^Tw$，梯度为 $X^T(\sigma(Xw)-y)+w$，Hessian为 $I+X^T\operatorname{diag}(p_i(1-p_i))X$。因为先验精度为1，即使三个特征共线，Hessian仍正定。这里反演**解析Hessian**，不把BFGS自带的近似逆矩阵当Laplace协方差。

每臂第五次反馈才重拟合，间隔内使用上一次后验，是计算开销折中。每条对话、每个算法种子重新实例化Selector。下格检查有限差分梯度/Hessian、MAP驻点、协方差、更新频率、Beta手算和未选奖励不影响更新；这比仅检查代码能运行更直接。

**局限：** Laplace为近似后验；TS和CTS模型族不同，差异不是纯特征因果消融；当前模板特征易于识别，无法据此外推真实对话；奖励即时且无噪声，不模拟LLM评价误差。

In [9]:
beta=Selector('TS',0)
for y0 in [1,1,0]: beta.update(2,y0,'What is its current dataset?')
assert (1+beta.s[2],1+beta.f[2])==(3,2)
assert (1+beta.s[2])/(2+beta.s[2]+beta.f[2])==.6
assert np.array_equal(beta.s[[0,1,3]],np.zeros(3))
q0='What is its current dataset?'; q1='What is P1 current metric?'
fit_test=Selector('LaplaceCTS',0)
for yy,qq in zip([1,0,1,0],[q0,q1,q0,q1]): fit_test.update(1,yy,qq)
assert np.array_equal(fit_test.mu,np.zeros((4,3)))  # 尚未第5次
fit_test.update(1,1,q0)
X=np.array(fit_test.X[1]); Y=np.array(fit_test.Y[1]); w=fit_test.mu[1]
def objective(w): return np.logaddexp(0,X@w).sum()-Y@(X@w)+.5*w@w
def gradient(w): return X.T@(expit(X@w)-Y)+w
eps=1e-5; I=np.eye(3)
fd=np.array([(objective(w+eps*e)-objective(w-eps*e))/(2*eps) for e in I])
p=expit(X@w); H=I+X.T@((p*(1-p))[:,None]*X)
fdH=np.column_stack([(gradient(w+eps*e)-gradient(w-eps*e))/(2*eps) for e in I])
assert np.allclose(fd,gradient(w),atol=1e-7)
assert np.allclose(fdH,H,atol=1e-7) and np.linalg.norm(gradient(w))<1e-3
assert np.allclose(H@fit_test.cov[1],I,atol=1e-8)
assert np.linalg.eigvalsh(fit_test.cov[1]).min()>0
assert np.array_equal(Selector('LaplaceCTS',0).mu,np.zeros((4,3)))
# 优化后的数组回放必须与原逐轮DataFrame实现一致，包括随机数调用顺序。
def reference_replay_actions(frame,name,seed):
    model=Selector(name,seed); actions=[]
    for _,group in frame.groupby('t',sort=True):
        group=group.set_index('arm').loc[ARMS]; q=group.question.iloc[0]
        k=model.choose(q,group.cost.to_numpy()); model.update(k,int(group.iloc[k].y),q)
        actions.append(ARMS[k])
    return actions
for name in ['TS','LaplaceCTS','RuleRouter']:
    assert reference_replay_actions(tiny,name,17)==replay(tiny,name,17).arm.tolist()
CHECKS['m3_optimized_replay_matches_reference']='passed on full development dialogue'
# 逐轮改变未选反事实奖励：同样已选反馈下，动作及后验必须相同。
for name in ['TS','LaplaceCTS']:
    a=Selector(name,73); b=Selector(name,73)
    for t0 in range(40):
        qq=q0 if t0%2 else q1; cc=np.array([120,510,260,270])
        ka=a.choose(qq,cc); kb=b.choose(qq,cc); assert ka==kb
        all_y=np.array([t0%2,1,0,1]); changed=1-all_y; changed[ka]=all_y[ka]
        a.update(ka,int(all_y[ka]),qq); b.update(kb,int(changed[kb]),qq)
        assert np.array_equal(a.s,b.s) and np.allclose(a.mu,b.mu) and np.allclose(a.cov,b.cov)
CHECKS['m3_beta_hand_calculation']='Beta(3,2), mean=0.6'
CHECKS['m3_gradient_hessian_covariance_refit_reset_feedback']='passed'
save_json('method_checks.json',{'checks':CHECKS,'map_gradient_norm':float(np.linalg.norm(gradient(w))),
    'hessian_fd_max_error':float(np.max(np.abs(fdH-H))),
    'covariance_min_eigenvalue':float(np.linalg.eigvalsh(fit_test.cov[1]).min())})
print('M3: Beta hand calculation, finite differences, posterior covariance, reset and feedback isolation passed.')

M3: Beta hand calculation, finite differences, posterior covariance, reset and feedback isolation passed.
